# S09: deploy an authenticated local API

70-minute lab. Exercise a real HTTP boundary with server-bound identities. The standard-library server is a single-process teaching deployment, not an internet-facing production server.

In [ ]:
from pathlib import Path
import os, sys, json, tempfile
# Run from repository root, notebooks/, or a Colab clone.
candidates=[Path.cwd(), Path.cwd().parent, Path('/content/ai-platform-architect-labs')]
ROOT=next((p for p in candidates if (p/'engine').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open the extracted repository. Colab: follow README Setup first.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–15 min | Inspect the boundary.** Three distinct generated credentials bind to three server-side application identities. Clients can send a question; they cannot select a tenant or application in the payload. The server defaults to loopback. Live mode disables the explicit query shortcut used for software rehearsal.

In [ ]:
import subprocess,secrets,socket,time,urllib.request,urllib.error
with socket.socket() as sock:
    sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
env=dict(os.environ,LAB_PORT=str(port),LAB_BIND='127.0.0.1',LIVE_MODE='0')
for name in ['POLICY_TOKEN','SPEND_TOKEN','BEACON_TOKEN']:env[name]=secrets.token_urlsafe(24)
process=subprocess.Popen([sys.executable,'-m','engine.server'],env=env,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
base=f'http://127.0.0.1:{port}'
for attempt in range(50):
    try:
        with urllib.request.urlopen(base+'/health',timeout=1) as r: print(json.load(r))
        break
    except urllib.error.URLError:time.sleep(.1)
else:
    process.terminate();raise RuntimeError('Local server did not become ready; check port/process setup.')
def request(token,body):
    req=urllib.request.Request(base+'/v1/ask',data=json.dumps(body).encode(),
        headers={'Content-Type':'application/json','Authorization':'Bearer '+token})
    try:
        with urllib.request.urlopen(req,timeout=10) as r:return r.status,json.load(r)
    except urllib.error.HTTPError as r:return r.code,json.load(r)

**15–35 min | Observe the deployed behavior.** Exercise authorised spend, invalid credentials and attempted tenant override. Each result comes from the actual local HTTP server. The sample below always runs search-and-SQL mode, even if other notebooks have live credentials.

In [ ]:
try:
    ok=request(env['SPEND_TOKEN'],{'question':'Nova Q1','query':{'quarter':'2026-Q1','supplier':'Nova'}})
    denied=request('wrong-token',{'question':'anything'})
    forged=request(env['POLICY_TOKEN'],{'question':'payment terms','tenant':'beacon'})
    assert ok[0]==200 and ok[1]['total_minor']==6500000
    assert denied[0]==401 and forged[0]==400
    print('Spend:',ok,'Invalid credential:',denied,'Tenant override:',forged)
finally:
    process.terminate();process.wait(timeout=5)
    process.stderr.close()

**35–55 min | Map to a client landing zone.** Replace static tokens with IdP-issued identities, validate issuer/audience/expiry, add TLS/ingress, secret management, workload identity, least-privilege data access, network egress controls and audit retention. Choose managed containers, serverless or Kubernetes only from workload and client constraints. Draw runtime, data, identity and management boundaries.

**55–70 min | Threat-review a changed design.** A vendor document contains instructions to reveal finance data. Explain why prompt instructions alone are insufficient, where source approval and ACLs apply, and how tool authority is constrained. Dockerfile is supplied for an optional local image build; Docker, cloud deployment, enterprise IAM and load testing are not demonstrated by this notebook. Record those validation gaps explicitly.